---
---

---

# TODO :

-   Mettre en place un Backend pour le profiling.
-   Utiliser ce backend avec le modèle LMTL complet (toutes les fonctions et production + biomasse)

---

---

---


# Notebook 4F : Décomposition du Temps de Calcul par Tâche

**Objectif** : Mesurer le temps passé dans chaque type de tâche (Transport, Production, Mortalité) pour identifier le goulot d'étranglement qui limite le speedup Strong Scaling.

## Hypothèse

Le transport (advection + diffusion) domine le temps de calcul car il traite des grilles 2D/3D volumineuses, contrairement aux calculs biologiques qui sont plus légers.


In [ ]:
import time
from collections import defaultdict
from datetime import datetime
from functools import wraps
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

print("✅ Imports réussis")

## Configuration


In [ ]:
CONFIG = {
    "grid_size": (500, 500),
    "n_cohorts": 12,
    "n_steps": 20,
    "n_groups": 1,  # Nombre de groupes fonctionnels
}

print(f"Configuration: {CONFIG}")

## Création des Données Synthétiques


In [ ]:
ny, nx = CONFIG["grid_size"]
n_cohorts = CONFIG["n_cohorts"]

# Créer les données
data = {
    # Variables 2D
    "temperature": xr.DataArray(np.random.rand(ny, nx) * 20 + 10, dims=["y", "x"]),
    "u": xr.DataArray(np.random.rand(ny, nx) * 0.2 - 0.1, dims=["y", "x"]),
    "v": xr.DataArray(np.random.rand(ny, nx) * 0.2 - 0.1, dims=["y", "x"]),
    "D": xr.DataArray(np.ones((ny, nx)) * 1000, dims=["y", "x"]),
    "dx": xr.DataArray(np.ones((ny, nx)) * 10000, dims=["y", "x"]),
    "dy": xr.DataArray(np.ones((ny, nx)) * 10000, dims=["y", "x"]),
    "cell_areas": xr.DataArray(np.ones((ny, nx)) * 1e8, dims=["y", "x"]),
    "mask": xr.DataArray(np.ones((ny, nx)), dims=["y", "x"]),
    # Variables 2D (biomasse)
    "biomass": xr.DataArray(np.random.rand(ny, nx) * 100, dims=["y", "x"]),
    # Variables 3D (production avec cohortes)
    "production": xr.DataArray(np.random.rand(n_cohorts, ny, nx) * 10, dims=["cohort", "y", "x"]),
}

print(f"Données créées:")
for k, v in data.items():
    print(f"  {k}: {v.shape}")

## Fonctions Simulées avec Profilage


In [ ]:
# Dictionnaire pour stocker les temps
timing_results = defaultdict(list)


def timed(category):
    """Décorateur pour mesurer le temps d'exécution."""

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            t_start = time.perf_counter()
            result = func(*args, **kwargs)
            t_end = time.perf_counter()
            timing_results[category].append(t_end - t_start)
            return result

        return wrapper

    return decorator


# ---------------------------------------------------------------------------
# TRANSPORT (le plus coûteux - simule advection + diffusion)
# ---------------------------------------------------------------------------
@timed("Transport Production")
def compute_transport_production(production, u, v, D, dx, dy, mask):
    """Simule le transport de production (3D: cohorts × y × x)."""
    # Simule un calcul coûteux sur toutes les cohortes
    result = production.copy()
    for _ in range(3):  # Simule advection + diffusion + boundary
        result = result * 0.99 + np.random.rand(*result.shape) * 0.01
    return {"transport_production_tendency": result}


@timed("Transport Biomass")
def compute_transport_biomass(biomass, u, v, D, dx, dy, mask):
    """Simule le transport de biomasse (2D: y × x)."""
    result = biomass.copy()
    for _ in range(3):
        result = result * 0.99 + np.random.rand(*result.shape) * 0.01
    return {"transport_biomass_tendency": result}


# ---------------------------------------------------------------------------
# BIOLOGIE (moins coûteux)
# ---------------------------------------------------------------------------
@timed("Production")
def compute_production(production, temperature):
    """Calcul de production (aging + recrutement)."""
    # Calcul simple
    result = production * 0.98
    return {"production_tendency": result}


@timed("Mortality")
def compute_mortality(biomass, temperature):
    """Calcul de mortalité."""
    rate = 0.001 * np.exp(0.1 * (temperature - 15))
    tendency = -rate * biomass
    return {"mortality_tendency": tendency}


# ---------------------------------------------------------------------------
# AUTRES
# ---------------------------------------------------------------------------
@timed("Time Integration")
def time_integrate(state, tendencies, dt=3600):
    """Intégration temporelle."""
    new_state = {}
    for key, value in state.items():
        if key in tendencies:
            new_state[key] = value + tendencies[key] * dt
        else:
            new_state[key] = value
    return new_state


print("✅ Fonctions créées avec profilage")

## Simulation avec Profilage


In [ ]:
# Réinitialiser les timings
timing_results.clear()

# Copier les données initiales
state = {k: v.copy() for k, v in data.items()}

print(f"Simulation de {CONFIG['n_steps']} pas de temps...")

t_total_start = time.perf_counter()

for step in range(CONFIG["n_steps"]):
    tendencies = {}

    # Simuler plusieurs groupes fonctionnels
    for group in range(CONFIG["n_groups"]):
        # Transport Production (le plus lourd)
        result = compute_transport_production(
            state["production"],
            state["u"],
            state["v"],
            state["D"],
            state["dx"],
            state["dy"],
            state["mask"],
        )
        tendencies.update(result)

        # Transport Biomasse
        result = compute_transport_biomass(
            state["biomass"],
            state["u"],
            state["v"],
            state["D"],
            state["dx"],
            state["dy"],
            state["mask"],
        )
        tendencies.update(result)

        # Production
        result = compute_production(state["production"], state["temperature"])
        tendencies.update(result)

        # Mortalité
        result = compute_mortality(state["biomass"], state["temperature"])
        tendencies.update(result)

    # Intégration temporelle
    state = time_integrate(state, tendencies)

t_total = time.perf_counter() - t_total_start

print(f"\n✅ Simulation terminée en {t_total:.3f}s")

## Analyse et Visualisation


In [ ]:
# Calculer les statistiques
summary = {}
for category, times in timing_results.items():
    summary[category] = {
        "total": sum(times),
        "mean": np.mean(times),
        "count": len(times),
    }

# Calculer les pourcentages
t_measured = sum(s["total"] for s in summary.values())
for category in summary:
    summary[category]["percent"] = summary[category]["total"] / t_measured * 100

# Afficher le tableau
print("\n" + "=" * 70)
print("DÉCOMPOSITION DU TEMPS DE CALCUL")
print("=" * 70)
print(f"{'Catégorie':<25} {'Temps (s)':<12} {'Appels':<10} {'%':<10}")
print("-" * 70)

# Trier par temps décroissant
sorted_categories = sorted(summary.items(), key=lambda x: x[1]["total"], reverse=True)

for category, stats in sorted_categories:
    print(f"{category:<25} {stats['total']:<12.3f} {stats['count']:<10} {stats['percent']:<10.1f}")

print("-" * 70)
print(f"{'TOTAL MESURÉ':<25} {t_measured:<12.3f} {'':<10} {'100.0':<10}")
print(f"{'TOTAL RÉEL':<25} {t_total:<12.3f}")
print("=" * 70)

# Calculer le % du transport
transport_percent = sum(s["percent"] for cat, s in summary.items() if "Transport" in cat)
print(f"\n📊 Transport total: {transport_percent:.1f}% du temps de calcul")

In [ ]:
# Créer la figure
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

categories = [cat for cat, _ in sorted_categories]
times = [s["total"] for _, s in sorted_categories]
percentages = [s["percent"] for _, s in sorted_categories]

# Couleurs : Transport en rouge/orange, autres en bleu/vert
colors = []
for cat in categories:
    if "Transport" in cat:
        colors.append("#E63946")  # Rouge
    elif "Production" in cat:
        colors.append("#457B9D")  # Bleu
    elif "Mortality" in cat:
        colors.append("#2A9D8F")  # Vert
    else:
        colors.append("#A8DADC")  # Gris-bleu

# --- Bar Chart ---
ax1 = axes[0]
bars = ax1.barh(categories, times, color=colors)
ax1.set_xlabel("Temps (s)")
ax1.set_title("Temps de Calcul par Catégorie")
ax1.invert_yaxis()

# Ajouter les pourcentages
for bar, pct in zip(bars, percentages):
    ax1.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.1f}%",
        va="center",
        fontsize=9,
    )

# --- Pie Chart ---
ax2 = axes[1]

# Grouper Transport
transport_time = sum(s["total"] for cat, s in summary.items() if "Transport" in cat)
other_times = {cat: s["total"] for cat, s in summary.items() if "Transport" not in cat}

pie_labels = ["Transport (total)"] + list(other_times.keys())
pie_values = [transport_time] + list(other_times.values())
pie_colors = ["#E63946", "#457B9D", "#2A9D8F", "#A8DADC"][: len(pie_labels)]

wedges, texts, autotexts = ax2.pie(
    pie_values, labels=pie_labels, autopct="%1.1f%%", colors=pie_colors, startangle=90
)
ax2.set_title("Répartition du Temps de Calcul")

plt.tight_layout()
plt.show()

# Sauvegarder
output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / "fig_04f_time_decomposition.png", dpi=300, bbox_inches="tight")
fig.savefig(output_dir / "fig_04f_time_decomposition.pdf", bbox_inches="tight")
print("\n✅ Figure sauvegardée")

## Implications pour le Strong Scaling


In [ ]:
# Calculer le speedup maximal selon Amdahl
f_sequential = transport_percent / 100  # Fraction du transport (dominante)
f_parallel = 1 - f_sequential

print("\n" + "=" * 70)
print("IMPLICATIONS POUR LE STRONG SCALING (Loi d'Amdahl)")
print("=" * 70)

print(f"\nFraction dominante (Transport) : {transport_percent:.1f}%")
print(f"Fraction parallélisable        : {f_parallel * 100:.1f}%")

print("\nSpeedup théorique maximal :")
print("-" * 40)
for n_workers in [2, 4, 8, 12, float("inf")]:
    if n_workers == float("inf"):
        speedup = 1 / f_sequential
        print(f"  ∞ workers : {speedup:.2f}× (limite)")
    else:
        speedup = 1 / (f_sequential + f_parallel / n_workers)
        print(f"  {int(n_workers):2} workers : {speedup:.2f}×")

print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)
print(f"""
Le transport représente {transport_percent:.0f}% du temps de calcul.
Avec une seule tâche de transport dominante, le speedup est limité par Amdahl.

RECOMMANDATIONS :
1. Paralléliser le transport lui-même (chunking spatial avec Dask Arrays)
2. Augmenter le nombre de groupes fonctionnels indépendants
3. Réduire l'overhead des wrappers Python/xarray
""")

## Génération du Résumé


In [ ]:
summary_text = f"""====================================================================================================
NOTEBOOK 4F: DÉCOMPOSITION DU TEMPS DE CALCUL PAR TÂCHE
====================================================================================================

DATE: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

OBJECTIF:
----------------------------------------------------------------------------------------------------
Identifier le goulot d'étranglement qui limite le speedup Strong Scaling
en mesurant le temps passé dans chaque type de tâche.

CONFIGURATION:
----------------------------------------------------------------------------------------------------
Grille                : {CONFIG["grid_size"][0]}×{CONFIG["grid_size"][1]}
Cohortes              : {CONFIG["n_cohorts"]}
Groupes fonctionnels  : {CONFIG["n_groups"]}
Pas de temps          : {CONFIG["n_steps"]}

RÉSULTATS:
----------------------------------------------------------------------------------------------------
{"Catégorie":<25} {"Temps (s)":<12} {"%":<10}
{"-" * 50}
"""

for category, stats in sorted_categories:
    summary_text += f"{category:<25} {stats['total']:<12.3f} {stats['percent']:<10.1f}\n"

summary_text += f"""
ANALYSE:
----------------------------------------------------------------------------------------------------
Transport total       : {transport_percent:.1f}% du temps de calcul
Speedup max (Amdahl)  : {1 / f_sequential:.2f}× avec ∞ workers

CONCLUSION:
----------------------------------------------------------------------------------------------------
Le transport domine le temps de calcul ({transport_percent:.0f}%).
Le speedup Strong Scaling est limité par la Loi d'Amdahl, pas par Dask.

RECOMMANDATIONS:
1. Paralléliser le transport avec chunking spatial (Dask Arrays)
2. Augmenter le nombre de groupes fonctionnels indépendants
3. Réduire l'overhead des wrappers Python/xarray

FICHIERS GÉNÉRÉS:
----------------------------------------------------------------------------------------------------
- fig_04f_time_decomposition.pdf/png
- notebook_04f_time_decomposition_summary.txt (ce fichier)

====================================================================================================
"""

# Sauvegarder
with open(output_dir / "notebook_04f_time_decomposition_summary.txt", "w") as f:
    f.write(summary_text)

print(summary_text)